## Fase 5 — Propuesta de Monitoreo

Una vez el modelo está sirviendo predicciones en producción (Fase 4), es necesario
vigilar si los datos que le llegan siguen pareciéndose a los datos con los que se
entrenó. Este notebook demuestra, con un caso real usando **Evidently AI**, cómo se
detectaría *data drift* en las lecturas de sensores — la señal de monitoreo más
temprana disponible, porque no requiere esperar a que se confirme si una máquina
falló o no (a diferencia de medir Recall/Precision reales, que sí tienen ese retraso).

Se generan dos reportes de comparación:
1. **Caso sano** — datos de entrenamiento vs. un lote de prueba normal (no debería
   mostrar drift significativo, es el comportamiento esperado).
2. **Caso con drift** — datos de entrenamiento vs. un lote simulado con un cambio
   realista en la distribución (para confirmar que Evidently sí lo detecta).

La propuesta completa de monitoreo (qué métricas, con qué frecuencia, y qué dispara
un reentrenamiento) queda documentada en `docs/06_propuesta_monitoreo.md`.

In [11]:
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent))

import pandas as pd
from evidently import Report
from evidently.presets import DataDriftPreset

from src.data.load_data import load_raw_data
from src.features.preprocessing import (
    get_feature_target_split,
    NUMERIC_FEATURES,
    CATEGORICAL_FEATURES,
)
from sklearn.model_selection import train_test_split

df = load_raw_data("../data/raw/ai4i2020.csv")
X, y = get_feature_target_split(df)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(X_train.shape, X_test.shape)
X_train.head()

(8000, 6) (2000, 6)


,Type,Air temperature [K],Process temperature [K],Rotational speed [rpm],Torque [Nm],Tool wear [min]
4058,M,302.0,310.9,1456,47.2,54
1221,M,297.0,308.3,1399,46.4,132
6895,M,301.0,311.6,1357,45.6,137
9863,L,298.9,309.8,1411,56.3,84
8711,L,297.1,308.5,1733,28.7,50


In [12]:
import pandas as pd

def resumen_drift(resultado) -> pd.DataFrame:
    """Convierte el resultado de un Report de Evidently en una tabla simple y liviana
    (en vez del widget interactivo completo, que es lo que estaba inflando el notebook)."""
    filas = []
    for m in resultado.dict()["metrics"]:
        config = m.get("config", {})
        if config.get("type") == "evidently:metric_v2:ValueDrift":
            score = float(m["value"])
            filas.append({
                "columna": config["column"],
                "metodo": config["method"],
                "umbral": config["threshold"],
                "score": round(score, 4),
                "drift_detectado": score > config["threshold"],
            })
    return pd.DataFrame(filas)

### Reporte 1 — Caso sano: entrenamiento vs. prueba

Evidently necesita que le digamos qué columnas son numéricas y cuáles categóricas
(`DataDefinition`), y luego arma un `Dataset` a partir del DataFrame con esa
definición. Comparamos `X_train` (referencia, lo que el modelo "conoce") contra
`X_test` — un lote que nunca vio, pero que viene de la **misma distribución**
original. Es el caso de control: si el pipeline de monitoreo está bien armado,
esto **no debería mostrar drift significativo**, porque `X_test` no es más que
otra muestra del mismo proceso.

In [13]:
from evidently import Dataset, DataDefinition

definition = DataDefinition(
    numerical_columns=NUMERIC_FEATURES,
    categorical_columns=CATEGORICAL_FEATURES,
)

reference_dataset = Dataset.from_pandas(X_train, data_definition=definition)
current_dataset_sano = Dataset.from_pandas(X_test, data_definition=definition)

report_sano = Report([DataDriftPreset()])
resultado_sano = report_sano.run(
    current_data=current_dataset_sano,
    reference_data=reference_dataset,
)
resumen_drift(resultado_sano)

,columna,metodo,umbral,score,drift_detectado
0,Air temperature [K],Wasserstein distance (normed),0.1,0.0157,False
1,Process temperature [K],Wasserstein distance (normed),0.1,0.0283,False
2,Rotational speed [rpm],Wasserstein distance (normed),0.1,0.0284,False
3,Torque [Nm],Wasserstein distance (normed),0.1,0.0268,False
4,Tool wear [min],Wasserstein distance (normed),0.1,0.0284,False
5,Type,Jensen-Shannon distance,0.1,0.0143,False


### Reporte 2 — Caso con drift simulado

Ahora simulamos un escenario realista de cambio operativo en la planta: la
línea empezó a producir casi exclusivamente piezas de calidad baja (`Type = "L"`)
y, al mismo tiempo, hubo un ajuste mecánico que corrió la lectura de temperatura
del aire unos grados hacia arriba. Ninguno de los dos cambios es catastrófico por
separado, pero juntos representan el tipo de deriva silenciosa que un modelo en
producción puede sufrir sin que nadie se dé cuenta — hasta que empieza a fallar
más de lo esperado.

In [14]:
X_test_con_drift = X_test.copy()
X_test_con_drift["Air temperature [K]"] = X_test_con_drift["Air temperature [K]"] + 6
X_test_con_drift["Type"] = "L"

current_dataset_drift = Dataset.from_pandas(X_test_con_drift, data_definition=definition)

report_drift = Report([DataDriftPreset()])
resultado_drift = report_drift.run(
    current_data=current_dataset_drift,
    reference_data=reference_dataset,
)
resumen_drift(resultado_drift)

,columna,metodo,umbral,score,drift_detectado
0,Air temperature [K],Wasserstein distance (normed),0.1,3.0036,True
1,Process temperature [K],Wasserstein distance (normed),0.1,0.0283,False
2,Rotational speed [rpm],Wasserstein distance (normed),0.1,0.0284,False
3,Torque [Nm],Wasserstein distance (normed),0.1,0.0268,False
4,Tool wear [min],Wasserstein distance (normed),0.1,0.0284,False
5,Type,Jensen-Shannon distance,0.1,0.4026,True


### Análisis

El reporte 1 confirma que el pipeline distingue bien entre variación normal y un
cambio real: comparando train vs. test (misma distribución), ninguna de las 6
columnas mostró drift, con distancias muy bajas (Wasserstein < 0.03 en todas).

El reporte 2, con el escenario simulado, sí detectó el cambio en las dos columnas
que efectivamente modificamos:
- `Air temperature [K]`: la distancia de Wasserstein pasó de 0.0157 a 3.0036 —
  un salto de casi 200x, señal inequívoca de drift.
- `Type`: la distancia de Jensen-Shannon pasó de 0.014 a 0.403 — coherente con
  haber forzado que el 100% de los registros fueran "L" en vez de la mezcla
  original L/M/H.

**Hallazgo clave para la propuesta de diseño:** aunque 2 de las 6 columnas
mostraron drift real y claro, el veredicto agregado de "Dataset Drift" siguió
diciendo **"NOT detected"** — porque Evidently solo marca el dataset completo
como "con drift" si más del 50% de las columnas lo muestran (`drift_share`,
umbral por defecto 0.5), y aquí solo fue el 33.3% (2 de 6). Es una lección real
de monitoreo en producción: **si un sistema de alertas solo mira el veredicto
agregado, se le puede escapar un drift real y significativo en una sola
variable importante** (como la temperatura, una de las features más influyentes
del modelo). Por eso la propuesta de monitoreo recomienda revisar también el
detalle por columna, no solo el resumen agregado, y usar un `drift_share` más
conservador en producción.

In [15]:
import os

os.makedirs("../docs/monitoring_reports", exist_ok=True)
resultado_sano.save_html("../docs/monitoring_reports/reporte_drift_sano.html")
resultado_drift.save_html("../docs/monitoring_reports/reporte_drift_simulado.html")

print("Reportes guardados en docs/monitoring_reports/")

Reportes guardados en docs/monitoring_reports/


In [16]:
resultado_dict = resultado_drift.dict()
print(list(resultado_dict.keys()))

['metrics', 'tests']


In [17]:
print(len(resultado_dict['metrics']))
print(resultado_dict['metrics'][0])

7
{'id': '15e89f895b482f9b84ba7274ed18a106', 'metric_name': 'DriftedColumnsCount(drift_share=0.5)', 'config': {'type': 'evidently:metric_v2:DriftedColumnsCount', 'drift_share': 0.5}, 'value': {'count': 2.0, 'share': 0.3333333333333333}}


In [18]:
print(resultado_dict['metrics'][1])

{'id': '55946edc24d91b05acd2df5464d07051', 'metric_name': 'ValueDrift(column=Air temperature [K],method=Wasserstein distance (normed),threshold=0.1)', 'config': {'type': 'evidently:metric_v2:ValueDrift', 'column': 'Air temperature [K]', 'method': 'Wasserstein distance (normed)', 'threshold': 0.1}, 'value': np.float64(3.0036270618877734)}
